
[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap03/cap03_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:20%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

# 3 Operações Espaciais: Histograma, Filtragem e Convolução

Este capítulo aborda o processamento de imagens diretamente no domínio espacial. Investigamos a manipulação de histogramas para realce de contraste, além do uso de filtros locais via convolução para suavização, redução de ruído e detecção de bordas.

## 3.1 Objetivos

Ao final deste capítulo, você será capaz de:

* **Manipular intensidade e pixels:** Executar operações aritméticas e lógicas (como subtração de fundo e máscaras bit a bit) e aplicar técnicas de realce por processamento de histograma (equalização e especificação);
* **Compreender fundamentos espaciais:** Entender os conceitos de vizinhança espacial, os mecanismos de convolução/correlação e o papel dos *kernels* (máscaras);
* **Aplicar filtragem espacial de suavização:** Utilizar filtros lineares de média e Gaussiano para redução de ruído e atenuação de detalhes;
* **Aplicar filtragem espacial de realce:** Dominar o uso de filtros de nitidez e bordas, como o Laplaciano, Sobel e a técnica de *Unsharp Masking*;
* **Utilizar filtros de ordem:** Aplicar o filtro de mediana para a remoção eficaz de ruídos específicos, como o ruído do tipo sal e pimenta;
* **Resolver problemas práticos:** Combinar essas técnicas de processamento e filtragem no pré-processamento de imagens para aplicações reais.

## 3.2 Configuração do Ambiente

In [ ]:
import os, importlib, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Baixar morph.py se necessário
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph
importlib.reload(morph)
from morph import mm

print("✅ Ambiente pronto")

In [ ]:
# Imagem de exemplo
url = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"
img_color = mm.read(url)
img = mm.gray(img_color)
print(f"Imagem carregada: {img.shape} | Tipo: {img.dtype}")

## 3.3 Introdução às Operações Espaciais

Uma operação espacial modifica o valor de um pixel considerando os valores de seus vizinhos.

Essas operações normalmente utilizam uma pequena matriz chamada:

- máscara;
- kernel;
- filtro;
- janela de convolução.

A ideia central consiste em percorrer a imagem aplicando operações locais sobre regiões vizinhas.

## 3.4 Manipulação de Intensidade e Pixels

Antes de entrarmos na filtragem espacial, é fundamental dominar operações ponto a ponto e o processamento de histogramas.

### 3.4.1 Histograma

O histograma representa a distribuição de intensidades de uma imagem.

In [ ]:
hist = cv2.calcHist([img], [0], None, [256], [0, 256])

plt.figure(figsize=(10, 4))
plt.plot(hist)
plt.title('Histograma')
plt.xlabel('Intensidade')
plt.ylabel('Frequência')
plt.show()

mm.show(img)

**Figura 3.1:** Histograma da imagem Lena em tons de cinza.


### 3.4.2 Equalização de Histograma

Técnica para realce de contraste que redistribui as intensidades.

In [ ]:
img_eq = cv2.equalizeHist(img)
mm.show([img, img_eq], titles=['Original', 'Equalizada'], cols=2)

**Figura 3.2:** Equalização de histograma.


## 3.5 Convolução Bidimensional

A convolução bidimensional é uma das operações mais importantes do PDI.

Seja:

$$
f(x,y)
$$

uma imagem de entrada e:

$$
w(s,t)
$$

um kernel de convolução.

A saída é dada por:

$$
g(x,y)=\sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)f(x-s,y-t)
$$

Essa operação permite implementar:
- suavização;
- realce;
- detecção de bordas;
- filtragem passa-baixa;
- filtragem passa-alta.

## 3.6 Kernels e Máscaras

Um kernel é uma pequena matriz utilizada para transformar uma imagem.

Exemplo de kernel de média:

$$
\frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

Esse filtro suaviza a imagem reduzindo variações locais.

In [ ]:
kernel_media = np.ones((3,3), dtype=np.float32) / 9
print(kernel_media)

## 3.7 Suavização por Média

In [ ]:
img_media = cv2.filter2D(img, -1, kernel_media)
mm.show([img, img_media], titles=['Original', 'Filtro de média'], cols=2)

**Figura 3.3:** Filtragem por média.


## 3.8 Filtro Gaussiano

In [ ]:
img_gauss = cv2.GaussianBlur(img, (5,5), 0)
mm.show([img, img_gauss], titles=['Original', 'Gaussiano'], cols=2)

**Figura 3.4:** Suavização Gaussiana.


## 3.9 Ruído em Imagens

In [ ]:
def add_salt_pepper(img, prob=0.05):
    noisy = img.copy()
    rnd = np.random.rand(*img.shape)
    noisy[rnd < prob/2] = 0
    noisy[rnd > 1 - prob/2] = 255
    return noisy

img_noise = add_salt_pepper(img)

In [ ]:
mm.show([img, img_noise], titles=['Original', 'Com ruído'], cols=2)

**Figura 3.5:** Imagem com ruído sal-e-pimenta.


## 3.10 Filtro da Mediana

In [ ]:
img_mediana = cv2.medianBlur(img_noise, 5)
mm.show([img_noise, img_mediana], titles=['Com ruído', 'Filtro mediana'], cols=2)

**Figura 3.6:** Remoção de ruído utilizando filtro da mediana.


## 3.11 Realce e Nitidez

In [ ]:
kernel_sharp = np.array([ [0, -1, 0], [-1, 5, -1], [0, -1, 0] ], dtype=np.float32)

In [ ]:
img_sharp = cv2.filter2D(img, -1, kernel_sharp)
mm.show([img, img_sharp], titles=['Original', 'Nitidez'], cols=2)

**Figura 3.7:** Realce de nitidez utilizando filtro passa-alta.


## 3.12 Detecção de Bordas

### 3.12.1 Gradiente e Operador de Sobel

In [ ]:
sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
magnitude = cv2.magnitude(sobelx, sobely)

mm.show([np.abs(sobelx), np.abs(sobely), magnitude], 
        titles=['Sobel X', 'Sobel Y', 'Magnitude'], cols=3)

**Figura 3.8:** Detecção de bordas utilizando Sobel.


### 3.12.2 Detector de Bordas de Canny

In [ ]:
edges = cv2.Canny(img, 100, 200)
mm.show(edges, title='Canny')

**Figura 3.9:** Detecção de bordas utilizando Canny.


### 3.12.3 Filtro Laplaciano

In [ ]:
lap = cv2.Laplacian(img, cv2.CV_64F)
mm.show(np.abs(lap), title='Laplaciano')

**Figura 3.10:** Filtro Laplaciano para detecção de bordas.


## 3.13 Comparação entre Filtros

## 3.14 Resumo

Neste capítulo foram apresentados:
- manipulação de histogramas e realce;
- convolução bidimensional;
- kernels espaciais;
- filtragem linear e não-linear;
- suavização (média, Gaussiano);
- redução de ruído (mediana);
- realce de nitidez e detecção de bordas (Sobel, Laplaciano, Canny).

Esses conceitos formam a base do pré-processamento em Visão Computacional.

## 3.15 🤖 Uso do NotebookLM como Tutor Complementar

Nesta edição, incentivamos o uso do **NotebookLM** como ferramenta complementar de aprendizagem.

[🚀 ACESSAR NOTEBOOKLM: CAPÍTULO 03](https://notebooklm.google.com/...) (substitua pelo link real quando disponível)

## 3.16 Lista de Exercícios

1. **(15%)** Implemente a convolução 2D manualmente usando NumPy (sem `cv2.filter2D`).

2. **(15%)** Compare os efeitos de filtros de média com diferentes tamanhos de kernel.

3. **(20%)** Adicione ruído sal-e-pimenta e compare os filtros de média, Gaussiano e mediana.

4. **(15%)** Aplique equalização de histograma e observe o efeito no contraste.

5. **(20%)** Implemente detecção de bordas com Sobel e Canny em uma imagem de sua escolha.

6. **(15%)** Explique a diferença entre correlação e convolução.

## Referências do Capítulo


* Gonzalez e Woods (2018) para fundamentos de filtragem espacial e histogramas.
* Bradski e Kaehler (2008) para implementação com OpenCV.

BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.